# 02 — Nettoyage et analyse exploratoire

**Objectif :** préparer les requêtes, clics et métadonnées produits pour tous les modèles.

**Entrées :** fichiers Kaggle dans `data/raw`, sinon corpus synthétique créé par le notebook 01.  
**Sorties :** CSV nettoyés dans `data/processed`, résumé JSON et figures dans `reports`.  
**Dépendance :** notebook 01.  
**Temps estimé :** moins de deux minutes pour le petit challenge.  
**Ressources :** CPU uniquement.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable.")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import (
    discover_competition_files,
    parse_products_xml,
    read_csv_as_strings,
)
from src.preprocessing import coerce_sku, normalize_query, normalize_text

CONFIG = yaml.safe_load((ROOT / "configs" / "default.yaml").read_text(encoding="utf-8"))
RAW_DIR = ROOT / CONFIG["paths"]["raw_data"]
PROCESSED_DIR = ROOT / CONFIG["paths"]["processed_data"]
REPORTS_DIR = ROOT / CONFIG["paths"]["reports"]
FIGURES_DIR = REPORTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Chargement de la source disponible

In [ ]:
competition = discover_competition_files(RAW_DIR)
if competition is not None:
    train = read_csv_as_strings(competition.train)
    test = read_csv_as_strings(competition.test)
    products = parse_products_xml(competition.products)
    source = "kaggle"
else:
    demo_dir = PROCESSED_DIR / "demo_source"
    required_demo = [demo_dir / "train.csv", demo_dir / "test.csv", demo_dir / "products.csv"]
    if not all(path.exists() for path in required_demo):
        raise FileNotFoundError("Exécutez d'abord le notebook 01 pour créer le corpus de secours.")
    train = read_csv_as_strings(required_demo[0])
    test = read_csv_as_strings(required_demo[1])
    products = read_csv_as_strings(required_demo[2])
    source = "synthetic-demo"

print(f"Source : {source}")
print(f"Train={train.shape}, Test={test.shape}, Produits={products.shape}")

## Normalisation et contrôles

In [ ]:
for frame in (train, test):
    frame["query"] = frame["query"].fillna("").astype(str)
    frame["query_text"] = frame["query"].map(normalize_text)
    frame["query_key"] = frame["query"].map(normalize_query)

train["sku"] = train["sku"].map(coerce_sku)
products["sku"] = products["sku"].map(coerce_sku)

for column in ("click_time", "query_time"):
    if column in train.columns:
        train[column] = pd.to_datetime(train[column], errors="coerce", utc=True)
if "query_time" in test.columns:
    test["query_time"] = pd.to_datetime(test["query_time"], errors="coerce", utc=True)

for column in ("title", "category", "description"):
    if column not in products.columns:
        products[column] = ""
    products[column] = products[column].fillna("").astype(str)

products["document"] = (
    products["title"] + " " + products["category"] + " " + products["description"]
).map(normalize_text)

train = train.loc[(train["sku"] != "") & (train["query_key"] != "")].copy()
test = test.loc[test["query_key"] != ""].copy()
products = products.loc[products["sku"] != ""].drop_duplicates("sku").copy()

assert train["sku"].map(type).eq(str).all()
assert train["query_key"].ne("").all()
assert products["sku"].is_unique

display(train.head(3))
display(products.head(3))

## Statistiques descriptives

In [ ]:
summary = {
    "source": source,
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "product_rows": int(len(products)),
    "unique_queries_raw": int(train["query"].nunique()),
    "unique_queries_normalized": int(train["query_key"].nunique()),
    "unique_clicked_skus": int(train["sku"].nunique()),
    "missing_query_time": int(train.get("query_time", pd.Series(dtype=object)).isna().sum()),
    "missing_click_time": int(train.get("click_time", pd.Series(dtype=object)).isna().sum()),
    "duplicate_click_rows": int(train.duplicated().sum()),
    "products_without_title": int(products["title"].eq("").sum()),
    "products_without_description": int(products["description"].eq("").sum()),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Visualisations compactes

In [ ]:
query_lengths = train["query_text"].str.split().map(len)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(query_lengths, bins=range(1, max(3, int(query_lengths.max()) + 2)), color="#107C10")
ax.set(title="Longueur des requêtes", xlabel="Nombre de mots", ylabel="Clics")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "query_length.png", dpi=140)
plt.close(fig)

if "category" in train.columns:
    categories = train["category"].replace("", "inconnue").value_counts().head(10)
    fig, ax = plt.subplots(figsize=(8, 4))
    categories.sort_values().plot.barh(ax=ax, color="#2D7D9A")
    ax.set(title="Catégories les plus cliquées", xlabel="Clics", ylabel="")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "top_categories.png", dpi=140)
    plt.close(fig)

print(f"Figures sauvegardées dans {FIGURES_DIR}")

## Sauvegarde des tables nettoyées

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train.to_csv(PROCESSED_DIR / "train_clean.csv", index=False)
test.to_csv(PROCESSED_DIR / "test_clean.csv", index=False)
products.to_csv(PROCESSED_DIR / "products_clean.csv", index=False)
(REPORTS_DIR / "eda_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Tables nettoyées sauvegardées.")

## Conclusion

Les requêtes possèdent deux représentations : `query_text` pour la recherche textuelle et
`query_key` pour la mémorisation exacte. Les SKU restent des chaînes de caractères.